In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report,
                             confusion_matrix)
import warnings
warnings.filterwarnings('ignore')


In [4]:
df = pd.read_csv("../data/processed/df_model_ready.csv")
print("=" * 60)
print("Dataset Loaded —", df.shape)
print(" ✅ Columns:", df.columns)

Dataset Loaded — (148835, 25)
 ✅ Columns: Index(['Duration', 'Impressions', 'Clicks', 'Leads', 'Conversions', 'Revenue',
       'Acquisition_Cost', 'ROI', 'Engagement_Score', 'Total_Cost',
       'Clicks_log', 'Leads_log', 'Conversions_log', 'Revenue_log',
       'Acquisition_Cost_log', 'Profit_Label', 'channel_email',
       'channel_facebook', 'channel_google', 'channel_instagram',
       'channel_whatsapp', 'channel_youtube', 'Campaign_Type_encoded',
       'Target_Audience_encoded', 'Customer_Segment_encoded'],
      dtype='str')


In [5]:
FEATURES = [
    'Conversions_log',
    'Leads',                  
    'Clicks',     
    'Impressions',
    'Acquisition_Cost_log',   
    'Engagement_Score',
    'Total_Cost',
    'Duration',
    'Campaign_Type_encoded',
    'Target_Audience_encoded',
    'Customer_Segment_encoded',
    'channel_email',
    'channel_facebook',
    'channel_google',
    'channel_instagram',
    'channel_whatsapp',
    'channel_youtube',
]

X = df[FEATURES]
y = df['Profit_Label']
 
print(f"\nInput & Output")
print(f"  Features : {len(FEATURES)} columns")
print(f"  Target   : Profit_Label  (1=Profit, 0=Loss)")
print(f"  Class balance:")
print(f"    Profit (1): {(y==1).sum():,}  ({(y==1).mean()*100:.1f}%)")
print(f"    Loss   (0): {(y==0).sum():,}  ({(y==0).mean()*100:.1f}%)")



Input & Output
  Features : 17 columns
  Target   : Profit_Label  (1=Profit, 0=Loss)
  Class balance:
    Profit (1): 113,005  (75.9%)
    Loss   (0): 35,830  (24.1%)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain-Test Split (80/20) — stratified")
print(f"  X_train: {X_train.shape}")
print(f"  X_test : {X_test.shape}")



Train-Test Split (80/20) — stratified
  X_train: (119068, 17)
  X_test : (29767, 17)


In [7]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    acc  = accuracy_score(y_te, pred)
    prec = precision_score(y_te, pred)
    rec  = recall_score(y_te, pred)
    f1   = f1_score(y_te, pred)
    print(f"\n  {name}")
    print(f"    Accuracy  : {acc:.4f}")
    print(f"    Precision : {prec:.4f}")
    print(f"    Recall    : {rec:.4f}")
    print(f"    F1-Score  : {f1:.4f}")
    print(f"\n    Classification Report:")
    print(classification_report(y_te, pred, target_names=['Loss (0)', 'Profit (1)']))
    print(f"    Confusion Matrix:")
    print(confusion_matrix(y_te, pred))
    return {"model": model, "Accuracy": acc, "Precision": prec,
            "Recall": rec, "F1": f1, "pred": pred}
 
print("\nSTEP 4: Training Models...")
 
results = {}
 
results["Logistic Regression"] = evaluate(
    "Logistic Regression",
    LogisticRegression(max_iter=1000, random_state=42),
    X_train, y_train, X_test, y_test
)
 
results["Decision Tree"] = evaluate(
    "Decision Tree",
    DecisionTreeClassifier(max_depth=10, min_samples_leaf=5, random_state=42),
    X_train, y_train, X_test, y_test
)
 
results["Random Forest"] = evaluate(
    "Random Forest",
    RandomForestClassifier(n_estimators=100, max_depth=15,
                           min_samples_leaf=3, random_state=42, n_jobs=-1),
    X_train, y_train, X_test, y_test
)


STEP 4: Training Models...

  Logistic Regression
    Accuracy  : 0.9091
    Precision : 0.9318
    Recall    : 0.9498
    F1-Score  : 0.9407

    Classification Report:
              precision    recall  f1-score   support

    Loss (0)       0.83      0.78      0.81      7166
  Profit (1)       0.93      0.95      0.94     22601

    accuracy                           0.91     29767
   macro avg       0.88      0.87      0.87     29767
weighted avg       0.91      0.91      0.91     29767

    Confusion Matrix:
[[ 5595  1571]
 [ 1134 21467]]

  Decision Tree
    Accuracy  : 0.9086
    Precision : 0.9265
    Recall    : 0.9554
    F1-Score  : 0.9407

    Classification Report:
              precision    recall  f1-score   support

    Loss (0)       0.84      0.76      0.80      7166
  Profit (1)       0.93      0.96      0.94     22601

    accuracy                           0.91     29767
   macro avg       0.89      0.86      0.87     29767
weighted avg       0.91      0.91      0

In [8]:
print("\n" + "=" * 60)
print("STEP 5: Model Comparison")
print(f"\n{'Model':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 68)
for name, res in results.items():
    print(f"{name:<25} {res['Accuracy']:>10.4f} {res['Precision']:>10.4f} "
          f"{res['Recall']:>10.4f} {res['F1']:>10.4f}")
 
best_name  = max(results, key=lambda k: results[k]['F1'])
best_model = results[best_name]['model']
print(f"\n  ✅ Best Model : {best_name}  (F1 = {results[best_name]['F1']:.4f})")



STEP 5: Model Comparison

Model                       Accuracy  Precision     Recall         F1
--------------------------------------------------------------------
Logistic Regression           0.9091     0.9318     0.9498     0.9407
Decision Tree                 0.9086     0.9265     0.9554     0.9407
Random Forest                 0.9125     0.9277     0.9596     0.9433

  ✅ Best Model : Random Forest  (F1 = 0.9433)


In [10]:
with open("parameters/classification_model.pkl", "wb") as f:
    pickle.dump(best_model, f)
 
with open("parameters/classification_features.pkl", "wb") as f:
    pickle.dump(FEATURES, f)
 
print("\nSTEP 6: Saved")
print("  → classification_model.pkl")
print("  → classification_features.pkl")
print("\n" + "=" * 60)
print("Classification Model Complete ✅")
print("=" * 60)



STEP 6: Saved
  → classification_model.pkl
  → classification_features.pkl

Classification Model Complete ✅
